# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('default')

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

BASELINE_TARGETS_DIR = PROJECT_ROOT / 'data' / 'processed'
FOLLOWUP_TARGETS_DIR = PROJECT_ROOT / 'data' / 'processed_follow_up'
IMAGES_DIR = PROJECT_ROOT / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

def load_targets_from_dir(target_dir: Path) -> pd.DataFrame:
    rows = []
    for file_path in sorted(target_dir.glob('CT_*_targets.csv')):
        match = re.match(r'(CT_\d+)_targets\.csv', file_path.name)
        if not match:
            continue

        subject_id = match.group(1)
        df = pd.read_csv(file_path)
        if df.empty:
            continue

        row = df.iloc[0].to_dict()
        row['subject_id'] = subject_id
        rows.append(row)

    if not rows:
        return pd.DataFrame(columns=['subject_id']).set_index('subject_id')

    return pd.DataFrame(rows).set_index('subject_id').sort_index()

baseline_targets = load_targets_from_dir(BASELINE_TARGETS_DIR)
followup_targets = load_targets_from_dir(FOLLOWUP_TARGETS_DIR)

print('Baseline targets shape:', baseline_targets.shape)
print('Follow-up targets shape:', followup_targets.shape)
print('Common subjects:', len(set(baseline_targets.index) & set(followup_targets.index)))
print('Images directory:', IMAGES_DIR)

In [ ]:
required_target = 'VO2peak_abs'

if required_target not in baseline_targets.columns:
    raise KeyError(f"{required_target} not found in baseline targets.")
if required_target not in followup_targets.columns:
    raise KeyError(f"{required_target} not found in follow-up targets.")

paired = (
    baseline_targets[[required_target]]
    .rename(columns={required_target: 'VO2peak_abs_pre'})
    .join(
        followup_targets[[required_target]].rename(columns={required_target: 'VO2peak_abs_post'}),
        how='inner'
    )
    .dropna()
)
paired['delta_vo2peak_abs'] = paired['VO2peak_abs_post'] - paired['VO2peak_abs_pre']

print('Paired sample size (non-missing):', len(paired))
paired

In [ ]:
if len(paired) < 2:
    raise ValueError('At least 2 paired observations are needed for inferential testing.')

pre = paired['VO2peak_abs_pre'].to_numpy(dtype=float)
post = paired['VO2peak_abs_post'].to_numpy(dtype=float)
delta = paired['delta_vo2peak_abs'].to_numpy(dtype=float)
n = len(delta)

mean_pre = pre.mean()
sd_pre = pre.std(ddof=1)
mean_post = post.mean()
sd_post = post.std(ddof=1)
mean_change = delta.mean()
sd_change = delta.std(ddof=1)
se_change = sd_change / np.sqrt(n)

ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=mean_change, scale=se_change)

ttest_res = stats.ttest_rel(post, pre)

try:
    wilcoxon_res = stats.wilcoxon(delta, zero_method='wilcox', alternative='two-sided', mode='auto')
    w_stat = wilcoxon_res.statistic
    w_p = wilcoxon_res.pvalue
except ValueError:
    w_stat = np.nan
    w_p = np.nan

if n >= 3:
    shapiro_res = stats.shapiro(delta)
    shapiro_stat = shapiro_res.statistic
    shapiro_p = shapiro_res.pvalue
else:
    shapiro_stat = np.nan
    shapiro_p = np.nan

cohens_dz = mean_change / sd_change if sd_change > 0 else np.nan

nonzero_delta = delta[delta != 0]
if len(nonzero_delta) > 0:
    abs_ranks = stats.rankdata(np.abs(nonzero_delta))
    w_plus = abs_ranks[nonzero_delta > 0].sum()
    w_minus = abs_ranks[nonzero_delta < 0].sum()
    rank_biserial = (w_plus - w_minus) / (w_plus + w_minus)
else:
    rank_biserial = np.nan

results_q1 = pd.DataFrame(
    [
        ['n_pairs', n],
        ['pre_mean', mean_pre],
        ['pre_sd', sd_pre],
        ['post_mean', mean_post],
        ['post_sd', sd_post],
        ['mean_change_post_minus_pre', mean_change],
        ['mean_change_95CI_low', ci_low],
        ['mean_change_95CI_high', ci_high],
        ['paired_t_statistic', ttest_res.statistic],
        ['paired_t_pvalue', ttest_res.pvalue],
        ['wilcoxon_statistic', w_stat],
        ['wilcoxon_pvalue', w_p],
        ['shapiro_statistic', shapiro_stat],
        ['shapiro_pvalue', shapiro_p],
        ['cohens_dz', cohens_dz],
        ['rank_biserial_correlation', rank_biserial],
    ],
    columns=['metric', 'value']
)

results_q1

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for subject_id, row in paired.iterrows():
    ax.plot([0, 1], [row['VO2peak_abs_pre'], row['VO2peak_abs_post']], marker='o', linewidth=1.2, alpha=0.7)

ax.plot([0, 1], [mean_pre, mean_post], marker='D', color='black', linewidth=2.5, label='Group mean')

ax.set_xticks([0, 1])
ax.set_xticklabels(['Pre', 'Post'])
ax.set_ylabel('VO2peak_abs (L/min)')
ax.set_title('Question 1: Paired pre-post VO2peak_abs change')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()

q1_figure_path = IMAGES_DIR / 'q1_vo2peak_abs_paired_pre_post.png'
fig.savefig(q1_figure_path, dpi=300, bbox_inches='tight')
print('Saved figure:', q1_figure_path)

plt.show()

In [ ]:
delta_series = paired['delta_vo2peak_abs'].copy()


heterogeneity_metrics = pd.DataFrame(
    [
        ['n_subjects', len(delta_series)],
        ['mean_delta', delta_series.mean()],
        ['sd_delta', delta_series.std(ddof=1)],
        ['median_delta', delta_series.median()],
        ['iqr_delta', delta_series.quantile(0.75) - delta_series.quantile(0.25)],
        ['min_delta', delta_series.min()],
        ['max_delta', delta_series.max()],
        ['range_delta', delta_series.max() - delta_series.min()],
    ],
    columns=['metric', 'value']
)


responder_counts = pd.Series(
    {
        'improved_delta_gt_0': int((delta_series > 0).sum()),
        'no_change_delta_eq_0': int((delta_series == 0).sum()),
        'decreased_delta_lt_0': int((delta_series < 0).sum()),
    }
)


responder_table = pd.DataFrame({
    'count': responder_counts,
    'percent': (responder_counts / len(delta_series) * 100).round(1)
})


print('Heterogeneity metrics:')
display(heterogeneity_metrics)


print('\nResponder profile:')
display(responder_table)

In [ ]:
delta_sorted = delta_series.sort_values()
colors = ['tab:green' if x > 0 else 'tab:red' if x < 0 else 'tab:gray' for x in delta_sorted.values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(delta_sorted.index, delta_sorted.values, color=colors)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Question 2: Waterfall of individual VO2peak_abs changes')
axes[0].set_ylabel('Delta VO2peak_abs (L/min)')
axes[0].tick_params(axis='x', rotation=90)
axes[0].grid(axis='y', alpha=0.3)

axes[1].hist(delta_series.values, bins=min(8, len(delta_series)), edgecolor='black', alpha=0.75, color='tab:blue')
axes[1].axvline(delta_series.mean(), color='black', linestyle='--', linewidth=1.5, label='Mean delta')
axes[1].set_title('Distribution of individual changes')
axes[1].set_xlabel('Delta VO2peak_abs (L/min)')
axes[1].set_ylabel('Count')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()

q2_figure_path = IMAGES_DIR / 'q2_vo2peak_abs_heterogeneity_waterfall_histogram.png'
fig.savefig(q2_figure_path, dpi=300, bbox_inches='tight')
print('Saved figure:', q2_figure_path)

plt.show()

In [ ]:
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

paired.to_csv(RESULTS_DIR / 'q1_q2_vo2peak_abs_paired_table.csv')
results_q1.to_csv(RESULTS_DIR / 'q1_vo2peak_abs_statistics.csv', index=False)
heterogeneity_metrics.to_csv(RESULTS_DIR / 'q2_vo2peak_abs_heterogeneity_metrics.csv', index=False)
responder_table.to_csv(RESULTS_DIR / 'q2_vo2peak_abs_responder_profile.csv')

print('Saved:')
print('-', RESULTS_DIR / 'q1_q2_vo2peak_abs_paired_table.csv')
print('-', RESULTS_DIR / 'q1_vo2peak_abs_statistics.csv')
print('-', RESULTS_DIR / 'q2_vo2peak_abs_heterogeneity_metrics.csv')
print('-', RESULTS_DIR / 'q2_vo2peak_abs_responder_profile.csv')

In [ ]:
pre_mean = float(results_q1.loc[results_q1['metric'] == 'pre_mean', 'value'].iloc[0])
pre_sd = float(results_q1.loc[results_q1['metric'] == 'pre_sd', 'value'].iloc[0])
post_mean = float(results_q1.loc[results_q1['metric'] == 'post_mean', 'value'].iloc[0])
post_sd = float(results_q1.loc[results_q1['metric'] == 'post_sd', 'value'].iloc[0])
delta_mean = float(results_q1.loc[results_q1['metric'] == 'mean_change_post_minus_pre', 'value'].iloc[0])
ci_low = float(results_q1.loc[results_q1['metric'] == 'mean_change_95CI_low', 'value'].iloc[0])
ci_high = float(results_q1.loc[results_q1['metric'] == 'mean_change_95CI_high', 'value'].iloc[0])
ttest_p = float(results_q1.loc[results_q1['metric'] == 'paired_t_pvalue', 'value'].iloc[0])
wilcoxon_p = float(results_q1.loc[results_q1['metric'] == 'wilcoxon_pvalue', 'value'].iloc[0])
shapiro_p = float(results_q1.loc[results_q1['metric'] == 'shapiro_pvalue', 'value'].iloc[0])
cohens_dz = float(results_q1.loc[results_q1['metric'] == 'cohens_dz', 'value'].iloc[0])
n_pairs = int(results_q1.loc[results_q1['metric'] == 'n_pairs', 'value'].iloc[0])

improved = int(responder_table.loc['improved_delta_gt_0', 'count'])
decreased = int(responder_table.loc['decreased_delta_lt_0', 'count'])
no_change = int(responder_table.loc['no_change_delta_eq_0', 'count'])
improved_pct = float(responder_table.loc['improved_delta_gt_0', 'percent'])
decreased_pct = float(responder_table.loc['decreased_delta_lt_0', 'percent'])

delta_min = float(heterogeneity_metrics.loc[heterogeneity_metrics['metric'] == 'min_delta', 'value'].iloc[0])
delta_max = float(heterogeneity_metrics.loc[heterogeneity_metrics['metric'] == 'max_delta', 'value'].iloc[0])
delta_sd = float(heterogeneity_metrics.loc[heterogeneity_metrics['metric'] == 'sd_delta', 'value'].iloc[0])
delta_iqr = float(heterogeneity_metrics.loc[heterogeneity_metrics['metric'] == 'iqr_delta', 'value'].iloc[0])

paragraph = (
    f"In {n_pairs} paired participants, VO2peak_abs increased from {pre_mean:.2f}±{pre_sd:.2f} L/min at baseline "
    f"to {post_mean:.2f}±{post_sd:.2f} L/min post-training, corresponding to a mean change of {delta_mean:.3f} L/min "
    f"(95% CI {ci_low:.3f} to {ci_high:.3f}). The pre-post effect was statistically significant in both the paired t-test "
    f"(p={ttest_p:.3f}) and Wilcoxon signed-rank test (p={wilcoxon_p:.3f}), with moderate paired effect size "
    f"(Cohen’s dz={cohens_dz:.2f}). The Shapiro-Wilk test on paired differences (p={shapiro_p:.3f}) did not indicate "
    f"a strong violation of normality. Individual response heterogeneity was evident: {improved}/{n_pairs} ({improved_pct:.1f}%) "
    f"participants improved, {decreased}/{n_pairs} ({decreased_pct:.1f}%) decreased, and {no_change}/{n_pairs} showed no change, "
    f"with changes spanning {delta_min:.3f} to {delta_max:.3f} L/min (SD {delta_sd:.3f}, IQR {delta_iqr:.3f})."
)

print(paragraph)